In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/creditcard.csv")  # adjust path to wherever your csv actually is
df.shape

(284807, 31)

In [2]:
df[["Time", "Amount", "Class"]].describe()

,Time,Amount,Class
count,284807.000000,284807.000000,284807.000000
mean,94813.859575,88.349619,0.001727
std,47488.145955,250.120109,0.041527
min,0.000000,0.000000,0.000000
25%,54201.500000,5.600000,0.000000
50%,84692.000000,22.000000,0.000000
75%,139320.500000,77.165000,0.000000
max,172792.000000,25691.160000,1.000000


In [3]:
split_time = df["Time"].median()  # rough halfway point in seconds
print("Split at:", split_time, "seconds (~", split_time / 3600, "hours )")

train_mask = df["Time"] < split_time
test_mask = ~train_mask

print("Train fraud rate:", df.loc[train_mask, "Class"].mean())
print("Test fraud rate:", df.loc[test_mask, "Class"].mean())
print("Train size:", train_mask.sum(), " Test size:", test_mask.sum())

Split at: 84692.0 seconds (~ 23.525555555555556 hours )
Train fraud rate: 0.0018890051473634685
Test fraud rate: 0.0015659672481110082
Train size: 142403  Test size: 142404


In [4]:
feature_cols = [c for c in df.columns if c not in ["Class"]]  # keep Time and Amount in for now, we'll decide on Time next

X_train = df.loc[train_mask, feature_cols].reset_index(drop=True)
y_train = df.loc[train_mask, "Class"].reset_index(drop=True)
X_test = df.loc[test_mask, feature_cols].reset_index(drop=True)
y_test = df.loc[test_mask, "Class"].reset_index(drop=True)

X_train.shape, X_test.shape

((142403, 30), (142404, 30))

In [5]:
def add_hour_feature(X, time_col="Time"):
    X = X.copy()
    X["Hour"] = (X[time_col] % 86400) // 3600  # seconds-in-day -> hour of day (0-23)
    X = X.drop(columns=[time_col])
    return X

X_train = add_hour_feature(X_train)
X_test = add_hour_feature(X_test)

X_train[["Hour"]].describe()

,Hour
count,142403.000000
mean,14.070708
std,5.857287
min,0.000000
25%,10.000000
50%,15.000000
75%,19.000000
max,23.000000


In [6]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos

print(f"Negative: {n_neg}, Positive: {n_pos}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

Negative: 142134, Positive: 269
scale_pos_weight: 528.38


In [7]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",   # PR-AUC, not accuracy — matches how we'll evaluate
    random_state=42,
)

model.fit(X_train, y_train)
print("Trained.")

Trained.


In [8]:
from sklearn.metrics import (
    classification_report,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
)

y_proba = model.predict_proba(X_test)[:, 1]  # probability of fraud
y_pred = (y_proba >= 0.5).astype(int)         # default 0.5 threshold, just as a first look

print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))
print("PR-AUC:", average_precision_score(y_test, y_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00    142181
       Fraud       0.56      0.83      0.67       223

    accuracy                           1.00    142404
   macro avg       0.78      0.91      0.83    142404
weighted avg       1.00      1.00      1.00    142404

PR-AUC: 0.7732874285686946
Confusion matrix:
 [[142038    143]
 [    39    184]]


In [9]:
import numpy as np

model_soft = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=np.sqrt(scale_pos_weight),  # ~23 instead of 528
    eval_metric="aucpr",
    random_state=42,
)
model_soft.fit(X_train, y_train)

y_proba_soft = model_soft.predict_proba(X_test)[:, 1]
y_pred_soft = (y_proba_soft >= 0.5).astype(int)

print(classification_report(y_test, y_pred_soft, target_names=["Legit", "Fraud"]))
print("PR-AUC:", average_precision_score(y_test, y_proba_soft))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_soft))

              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00    142181
       Fraud       0.63      0.82      0.71       223

    accuracy                           1.00    142404
   macro avg       0.81      0.91      0.86    142404
weighted avg       1.00      1.00      1.00    142404

PR-AUC: 0.8081147281450363
Confusion matrix:
 [[142073    108]
 [    40    183]]


In [10]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_soft)

# Find threshold that gets recall closest to, say, 90% — catching more fraud
# is often worth more false positives in a fraud context, but you should judge that
target_recall = 0.90
idx = np.argmin(np.abs(recalls - target_recall))
print(f"At recall={recalls[idx]:.2f}: precision={precisions[idx]:.2f}, threshold={thresholds[idx]:.3f}")

# Also show a few other points along the curve for comparison
for r in [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]:
    i = np.argmin(np.abs(recalls - r))
    print(f"recall={recalls[i]:.2f}  precision={precisions[i]:.2f}  threshold={thresholds[i] if i < len(thresholds) else 'N/A':.3f}")

At recall=0.90: precision=0.04, threshold=0.000
recall=0.70  precision=0.92  threshold=0.992
recall=0.75  precision=0.85  threshold=0.970
recall=0.80  precision=0.69  threshold=0.716
recall=0.85  precision=0.29  threshold=0.022
recall=0.90  precision=0.04  threshold=0.000
recall=0.95  precision=0.01  threshold=0.000


In [11]:
DEPLOYED_THRESHOLD = 0.970  # recall ~0.75, precision ~0.85 on held-out test set

In [13]:
import json
from sklearn.metrics import precision_score, recall_score

y_pred_deployed = (y_proba_soft >= DEPLOYED_THRESHOLD).astype(int)

metrics = {
    "dataset": {
        "total_transactions": int(len(df)),
        "fraud_transactions": int(df["Class"].sum()),
        "fraud_rate_pct": round(float(df["Class"].mean()) * 100, 4),
    },
    "model": {
        "algorithm": "XGBoost (supervised) + Isolation Forest (secondary signal)",
        "deployed_threshold": DEPLOYED_THRESHOLD,
        "precision": round(precision_score(y_test, y_pred_deployed), 3),
        "recall": round(recall_score(y_test, y_pred_deployed), 3),
        "pr_auc": round(average_precision_score(y_test, y_proba_soft), 3),
        "test_set_size": int(len(y_test)),
        "fraud_caught": int(((y_test == 1) & (y_pred_deployed == 1)).sum()),
        "fraud_missed": int(((y_test == 1) & (y_pred_deployed == 0)).sum()),
        "false_alarms": int(((y_test == 0) & (y_pred_deployed == 1)).sum()),
    },
}

with open("model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))

{
  "dataset": {
    "total_transactions": 284807,
    "fraud_transactions": 492,
    "fraud_rate_pct": 0.1727
  },
  "model": {
    "algorithm": "XGBoost (supervised) + Isolation Forest (secondary signal)",
    "deployed_threshold": 0.97,
    "precision": 0.856,
    "recall": 0.749,
    "pr_auc": 0.808,
    "test_set_size": 142404,
    "fraud_caught": 167,
    "fraud_missed": 56,
    "false_alarms": 28
  }
}


In [12]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.0018,
    random_state=42,
    n_jobs=-1,
)
iso_forest.fit(X_train_scaled)

iso_scores_test = iso_forest.decision_function(X_test_scaled)
iso_pred_test = iso_forest.predict(X_test_scaled)

In [13]:
iso_is_anomaly = (iso_pred_test == -1)
print(classification_report(y_test, iso_is_anomaly.astype(int), target_names=["Legit", "Fraud"]))

              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00    142181
       Fraud       0.19      0.20      0.20       223

    accuracy                           1.00    142404
   macro avg       0.60      0.60      0.60    142404
weighted avg       1.00      1.00      1.00    142404



In [14]:
import json

# Demo sample: deliberately over-represents fraud so a live demo actually
# shows flagged transactions within a reasonable time, instead of the real
# ~0.17% rate meaning you might watch for 10+ minutes and see nothing.
# This is a UX choice for demoing, not a claim about real-world fraud rates.

fraud_rows = X_test[y_test == 1].sample(n=40, random_state=7)
legit_rows = X_test[y_test == 0].sample(n=200, random_state=7)

demo_sample = pd.concat([fraud_rows, legit_rows]).sample(frac=1, random_state=7)  # shuffle
demo_records = demo_sample.reset_index(drop=True).to_dict(orient="records")

with open("live_sample.json", "w") as f:
    json.dump(demo_records, f)

actual_fraud_rate = len(fraud_rows) / len(demo_records)
print(f"Saved {len(demo_records)} transactions, {len(fraud_rows)} fraud ({actual_fraud_rate:.1%})")

Saved 240 transactions, 40 fraud (16.7%)


In [15]:
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType
import onnx

n_features = X_train.shape[1]
initial_type = [("float_input", FloatTensorType([None, n_features]))]

# Strip named features so onnxmltools' converter (which expects f0, f1, ...)
# doesn't choke on names like 'V14' — purely a conversion-time fix, doesn't
# change the model itself
booster = model_soft.get_booster()
booster.feature_names = None

onnx_xgb = convert_xgboost(model_soft, initial_types=initial_type)
onnx.save_model(onnx_xgb, "fraud_xgb.onnx")
print("Saved fraud_xgb.onnx with", n_features, "features")
print("Feature order:", X_train.columns.tolist())

Saved fraud_xgb.onnx with 30 features
Feature order: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Hour']


In [19]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType as SkFloatTensorType
import joblib

onnx_iso = convert_sklearn(
    iso_forest,
    initial_types=[("float_input", SkFloatTensorType([None, n_features]))],
    target_opset={"": 12, "ai.onnx.ml": 3},  # pin ai.onnx.ml to a version this skl2onnx build supports
)
onnx.save_model(onnx_iso, "iso_forest.onnx")

joblib.dump(scaler, "scaler.pkl")
print("Saved iso_forest.onnx and scaler.pkl")

Saved iso_forest.onnx and scaler.pkl


In [20]:
import onnxruntime as ort

# XGBoost ONNX
sess_xgb = ort.InferenceSession("fraud_xgb.onnx")
input_name = sess_xgb.get_inputs()[0].name

X_test_np = X_test.values.astype(np.float32)
onnx_outputs = sess_xgb.run(None, {input_name: X_test_np})

# XGBoost's ONNX classifier output is typically [labels, probabilities]
onnx_labels = onnx_outputs[0]
onnx_proba = np.array([d[1] for d in onnx_outputs[1]])  # probability of class 1 (fraud)

# Compare to original sklearn-API predictions
sklearn_proba = model_soft.predict_proba(X_test)[:, 1]

max_diff = np.abs(onnx_proba - sklearn_proba).max()
print("Max probability difference (ONNX vs sklearn):", max_diff)

onnx_pred_at_threshold = (onnx_proba >= DEPLOYED_THRESHOLD).astype(int)
sklearn_pred_at_threshold = (sklearn_proba >= DEPLOYED_THRESHOLD).astype(int)
print("Predictions match at deployed threshold:", (onnx_pred_at_threshold == sklearn_pred_at_threshold).all())

Max probability difference (ONNX vs sklearn): 2.9802322e-07
Predictions match at deployed threshold: True


In [21]:
sess_iso = ort.InferenceSession("iso_forest.onnx")
iso_input_name = sess_iso.get_inputs()[0].name

X_test_scaled_np = X_test_scaled.astype(np.float32)
iso_onnx_outputs = sess_iso.run(None, {iso_input_name: X_test_scaled_np})

print("Output count:", len(iso_onnx_outputs))
for i, out in enumerate(iso_onnx_outputs):
    print(f"Output {i} shape:", np.array(out).shape, "sample:", np.array(out)[:5])

Output count: 2
Output 0 shape: (142404, 1) sample: [[1]
 [1]
 [1]
 [1]
 [1]]
Output 1 shape: (142404, 1) sample: [[0.29388544]
 [0.28992307]
 [0.00369364]
 [0.28910437]
 [0.28057164]]
